# Analytics — ตอบคำถามธุรกิจกลุ่ม A

โหลด **dataset #1 (star schema)** จาก Hugging Face แล้วตอบคำถาม A1–A5 ใน
[`docs/business_questions.md`](../docs/business_questions.md) ทั้ง 5 ข้อ

ทุกคำถามในนี้ตอบได้ด้วยการนับและค่าเฉลี่ย **ไม่ต้องสร้างโมเดล** — ถ้าอยากดูฝั่งที่
ต้องใช้โมเดล ไปที่ [`../model/model.ipynb`](../model/model.ipynb)

> ใช้ DuckDB query ตรงบนไฟล์ Parquet แล้วดึงเฉพาะ**ผลสรุป**เข้า pandas
> ข้อมูลมี 3.66 ล้านแถว ถ้าโหลดเข้า pandas ทั้งก้อนจะกิน RAM เกินจำเป็น

In [ ]:
import sys
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from elt.common import load_config, resolve_dataset  # noqa: E402

config = load_config()
SRC = resolve_dataset("elt", config)          # ดึงจาก HF (cache ไว้ ไม่โหลดซ้ำ)

con = duckdb.connect()
con.execute(f"CREATE VIEW fact_review AS SELECT * FROM '{SRC}/fact_review/**/*.parquet'")
con.execute(f"CREATE VIEW review_text AS SELECT * FROM '{SRC}/review_text/**/*.parquet'")
con.execute(f"CREATE VIEW dim_product AS SELECT * FROM '{SRC}/dim_product.parquet'")
con.execute(f"CREATE VIEW dim_user   AS SELECT * FROM '{SRC}/dim_user.parquet'")
con.execute(f"CREATE VIEW dim_date   AS SELECT * FROM '{SRC}/dim_date.parquet'")

plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

con.sql("""
    SELECT category, count(*) AS n_reviews,
           count(DISTINCT product_key) AS n_products,
           round(avg(rating), 2) AS avg_rating
    FROM fact_review GROUP BY 1 ORDER BY 2 DESC
""").df()

## A1. แบรนด์ไหนแข็งแกร่งที่สุดในแต่ละหมวด?

**ใช้ตอบว่า** ควรใช้ใครเป็น benchmark และคู่แข่งตัวจริงคือใคร

`HAVING count(*) >= 200` สำคัญมาก — ถ้าไม่ใส่ แบรนด์ที่มีรีวิวเดียว 5 ดาว
จะขึ้นอันดับ 1 ทุกครั้ง

In [ ]:
brands = con.sql("""
    SELECT p.category, p.store AS brand,
           count(*) AS n_reviews,
           count(DISTINCT p.product_key) AS n_products,
           round(avg(f.rating), 2) AS avg_rating,
           round(avg(f.is_negative::INT), 3) AS negative_share
    FROM fact_review f JOIN dim_product p USING (product_key)
    WHERE p.store IS NOT NULL
    GROUP BY 1, 2
    HAVING count(*) >= 200
    QUALIFY row_number() OVER (PARTITION BY p.category ORDER BY avg(f.rating) DESC) <= 5
    ORDER BY p.category, avg_rating DESC
""").df()
brands

In [ ]:
# แบรนด์ที่รีวิวเยอะแต่คะแนนต่ำ = ปัญหาที่กระทบลูกค้าจำนวนมาก น่าสนใจกว่าอันดับต้น
con.sql("""
    SELECT p.category, p.store AS brand, count(*) AS n_reviews,
           round(avg(f.rating), 2) AS avg_rating,
           round(avg(f.is_negative::INT), 3) AS negative_share
    FROM fact_review f JOIN dim_product p USING (product_key)
    WHERE p.store IS NOT NULL
    GROUP BY 1, 2 HAVING count(*) >= 2000
    ORDER BY avg_rating ASC LIMIT 10
""").df()

## A2. ราคาแพงขึ้นแล้วลูกค้าพอใจขึ้นจริงไหม?

⚠️ **อ่านก่อน** — ชุดนี้รู้ราคาแค่ ~21% ของรีวิว และการขาดหาย**ไม่ได้สุ่ม**
cell แรกจึงวัดขนาดของอคติก่อน แล้วค่อยวิเคราะห์

In [ ]:
bias = con.sql("""
    SELECT CASE WHEN p.price IS NULL THEN 'ไม่รู้ราคา' ELSE 'รู้ราคา' END AS grp,
           count(*) AS n_reviews,
           round(count(*) * 100.0 / sum(count(*)) OVER (), 1) AS pct,
           round(avg(f.rating), 3) AS avg_rating
    FROM fact_review f JOIN dim_product p USING (product_key)
    GROUP BY 1
""").df()
display(bias)

gap = bias.loc[bias.grp == "รู้ราคา", "avg_rating"].iloc[0] - \
      bias.loc[bias.grp == "ไม่รู้ราคา", "avg_rating"].iloc[0]
print(f"ช่องว่างคะแนน: {gap:+.3f} ดาว")
print("→ สินค้าที่รู้ราคาได้คะแนนสูงกว่า การกรอง Unknown ทิ้งจึงทำให้ผลเอนบวก")
print("   ต้องรายงานข้อจำกัดนี้ทุกครั้งที่อ้างผลจาก A2")

In [ ]:
price = con.sql("""
    SELECT p.category, p.price_band,
           count(*) AS n_reviews,
           round(avg(f.rating), 3) AS avg_rating,
           round(avg(f.is_negative::INT), 3) AS negative_share
    FROM fact_review f JOIN dim_product p USING (product_key)
    WHERE p.price_band <> 'Unknown'
    GROUP BY 1, 2
""").df()

order = ["Budget (<$10)", "Low ($10-25)", "Mid ($25-50)", "High ($50-100)", "Premium ($100+)"]
pivot = price.pivot(index="price_band", columns="category", values="avg_rating").reindex(order)
display(pivot.round(3))

pivot.plot(marker="o")
plt.title("คะแนนเฉลี่ยตามช่วงราคา (เฉพาะสินค้าที่รู้ราคา ~21% ของรีวิว)")
plt.ylabel("คะแนนเฉลี่ย"); plt.xlabel("")
plt.xticks(range(len(order)), order, rotation=20, ha="right")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

## A3. สินค้าไหนกำลังขาลง?

คะแนนเฉลี่ยรวมขยับช้ามากเพราะถูกถ่วงด้วยรีวิวเก่า สินค้าที่คุณภาพเพิ่งแย่ลงจะยังดูดีอยู่
การเทียบ "ช่วงหลัง vs ก่อนหน้า" ทำให้เห็นปัญหาก่อนคะแนนรวมจะตก

⚠️ ใช้ **2020–2021** เป็นช่วงหลัง ไม่ใช่ 2022+ เพราะปริมาณรีวิวปี 2022–2023
เก็บไม่ครบ (ดูกราฟใน A5)

In [ ]:
declining = con.sql("""
    WITH per_product AS (
        SELECT f.product_key,
               avg(f.rating)                                   AS lifetime_avg,
               count(*)                                        AS lifetime_n,
               avg(f.rating) FILTER (WHERE d.year IN (2020, 2021)) AS recent_avg,
               count(*)      FILTER (WHERE d.year IN (2020, 2021)) AS recent_n
        FROM fact_review f JOIN dim_date d USING (date_key)
        GROUP BY f.product_key
    )
    SELECT p.category, p.store,
           substr(p.product_title, 1, 45) AS product,
           pp.lifetime_n, round(pp.lifetime_avg, 2) AS lifetime_avg,
           pp.recent_n,   round(pp.recent_avg, 2)   AS recent_avg,
           round(pp.recent_avg - pp.lifetime_avg, 2) AS delta
    FROM per_product pp JOIN dim_product p USING (product_key)
    WHERE pp.recent_n >= 30 AND pp.lifetime_n >= 100
    ORDER BY delta ASC LIMIT 15
""").df()
declining

## A4. รีวิวแบบไหนที่คนบอกว่ามีประโยชน์?

**ใช้ตอบว่า** ควรจัดลำดับรีวิวบนหน้าสินค้าอย่างไร และควรกระตุ้นให้ลูกค้าเขียนแบบไหน

ดู**ค่ามัธยฐาน**ควบคู่ค่าเฉลี่ยเสมอ เพราะการแจกแจงเบ้ขวาหนัก (ส่วนใหญ่ได้ 0 โหวต)

In [ ]:
helpful = con.sql("""
    SELECT f.has_images,
           CASE WHEN length(t.review_text) < 100 THEN '1. สั้น (<100)'
                WHEN length(t.review_text) < 500 THEN '2. กลาง (100-500)'
                ELSE                                  '3. ยาว (500+)' END AS length_bucket,
           count(*) AS n_reviews,
           round(avg(f.helpful_vote), 2)            AS avg_helpful,
           median(f.helpful_vote)                   AS median_helpful,
           round(avg((f.helpful_vote > 0)::INT), 3) AS share_with_any_vote
    FROM fact_review f JOIN review_text t USING (review_id)
    GROUP BY 1, 2 ORDER BY 2, 1
""").df()
display(helpful)

ax = helpful.pivot(index="length_bucket", columns="has_images",
                   values="share_with_any_vote").plot(kind="bar")
ax.set_title("สัดส่วนรีวิวที่ได้โหวต 'มีประโยชน์' อย่างน้อย 1 เสียง")
ax.set_ylabel("สัดส่วน"); ax.set_xlabel("")
ax.legend(title="แนบรูป"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

In [ ]:
# helpful_vote มีอคติเรื่องอายุ — รีวิวเก่ามีเวลาสะสมโหวตมากกว่า
# ถ้าไม่คุมตัวแปรนี้ จะสรุปผิดว่า "รีวิวยาวดีกว่า" ทั้งที่จริงแค่ "รีวิวเก่ากว่า"
con.sql("""
    SELECT d.year, count(*) AS n_reviews,
           round(avg(f.helpful_vote), 2) AS avg_helpful,
           round(avg((f.helpful_vote > 0)::INT), 3) AS share_with_any_vote
    FROM fact_review f JOIN dim_date d USING (date_key)
    WHERE d.year BETWEEN 2010 AND 2023
    GROUP BY 1 ORDER BY 1
""").df()

## A5. สามหมวดต่างกันแค่ไหน และคะแนนของใครน่าเชื่อถือน้อยที่สุด?

เราจงใจเลือก 3 หมวดที่รวมเป็นหมวดใหญ่เดียวกันได้ ข้อนี้จึงตรวจว่า
"เอามารวมกันวิเคราะห์ได้จริงไหม" — ถ้าสัดส่วน verified หรือความแตกขั้วต่างกันมาก
ห้ามเอาคะแนนข้ามหมวดมาเทียบกันตรง ๆ

In [ ]:
cats = con.sql("""
    SELECT f.category,
           count(*) AS n_reviews,
           round(avg(f.rating), 2)                 AS avg_rating,
           round(avg(f.verified_purchase::INT), 3) AS verified_share,
           round(avg((f.rating IN (1, 5))::INT), 3) AS polarized_share,
           round(avg(f.has_images::INT), 4)        AS with_images_share,
           round(avg((u.reviewer_segment = 'One-off')::INT), 3) AS oneoff_share
    FROM fact_review f JOIN dim_user u USING (user_key)
    GROUP BY 1 ORDER BY 2 DESC
""").df()
cats

In [ ]:
# ปริมาณรีวิวรายปี — เห็นชัดว่าปี 2022-2023 เก็บไม่ครบ ห้ามอ่านเป็น "ดีมานด์หด"
trend = con.sql("""
    SELECT d.year, f.category, count(*) AS n_reviews, round(avg(f.rating), 3) AS avg_rating
    FROM fact_review f JOIN dim_date d USING (date_key)
    WHERE d.year BETWEEN 2012 AND 2023
    GROUP BY 1, 2 ORDER BY 1
""").df()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
trend.pivot(index="year", columns="category", values="n_reviews").plot(ax=ax1, marker="o")
ax1.axvspan(2021.5, 2023, color="red", alpha=0.12)
ax1.set_title("จำนวนรีวิว (แถบแดง = ข้อมูลเก็บไม่ครบ)")
ax1.set_ylabel("จำนวนรีวิว"); ax1.legend(fontsize=7)

trend.pivot(index="year", columns="category", values="avg_rating").plot(ax=ax2, marker="o")
ax2.set_title("คะแนนเฉลี่ย (ตัวชี้วัดที่ไม่ขึ้นกับจำนวน — ใช้ดูแนวโน้มได้)")
ax2.set_ylabel("คะแนนเฉลี่ย"); ax2.legend(fontsize=7)
plt.tight_layout(); plt.show()

## สรุปสิ่งที่ได้

รันแล้วจะเห็นว่า:

- **A1** แบรนด์ที่รีวิวเยอะแต่คะแนนต่ำน่าสนใจกว่าอันดับต้น เพราะกระทบลูกค้าจำนวนมาก
- **A2** คะแนนมีแนวโน้มเพิ่มตามช่วงราคา แต่ข้อสรุปนี้อ้างได้เฉพาะสินค้าที่รู้ราคา (~21%)
  ซึ่งเป็นกลุ่มที่คะแนนสูงกว่าค่าเฉลี่ยอยู่แล้ว
- **A3** ได้รายชื่อสินค้าที่คะแนนช่วงหลังตกจากค่าเฉลี่ยตลอดชีพ → ส่งทีมตรวจสอบ
- **A4** รีวิวยาวและมีรูปได้โหวตมากกว่า แต่ต้องคุมอคติเรื่องอายุรีวิวก่อนสรุป
- **A5** ทั้ง 3 หมวดมี verified สูงใกล้เคียงกัน (90–94%) เทียบข้ามหมวดได้พอสมควร

ขั้นต่อไป — คำถามที่ต้องใช้โมเดล อยู่ที่ [`../model/model.ipynb`](../model/model.ipynb)

In [ ]:
con.close()
print("analytics เสร็จเรียบร้อย")